In [1]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, f1_score, roc_auc_score
import sys
from pathlib import Path

In [2]:
current_dir = Path.cwd()
utils_path = next(
    (
        p
        for p in [current_dir] + list(current_dir.parents)
        if (p / "notebook_utils.py").exists()
    ),
    None,
)

sys.path.append(str(utils_path))

In [3]:
from notebook_utils import (
    setup_env,
    load_data_for_modeling,
    get_exp_manager,
    save_sklearn_model,
)

CURRENT_STAGE = "01_baseline" 

exp_manager = get_exp_manager()
PROJECT_ROOT, config = setup_env()
df = load_data_for_modeling(config, PROJECT_ROOT, data_source="szfo_df")
target_col = config["features"]["target_col"]


Загрузка данных из: D:\Education\Arcticle\dtp_project\data\processed\dtp_szfo.parquet
Режим таргета: binary_severe. Распределение:
target
0    0.6065
1    0.3935
Name: proportion, dtype: float64


In [4]:
paths = exp_manager.get_paths(model_name="01_baseline")
print(f"Сохраняем модель сюда: {paths['model']}")
print(f"Сохраняем графики сюда: {paths['report']}")


Сохраняем модель сюда: D:\Education\Arcticle\dtp_project\res\01_dtp_stat\models\01_baseline
Сохраняем графики сюда: D:\Education\Arcticle\dtp_project\res\01_dtp_stat\reports\01_baseline


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=[target_col]), 
    df[target_col], 
    test_size=0.2, 
    random_state=42, 
    stratify=df[target_col]
)

In [6]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
save_sklearn_model(dummy, exp_manager, model_name='Dummy', stage=CURRENT_STAGE)

Модель сохранена: D:\Education\Arcticle\dtp_project\res\01_dtp_stat\models\01_baseline\Dummy\model.joblib


In [7]:
y_pred = dummy.predict(X_test)
y_proba = dummy.predict_proba(X_test)

In [8]:
print(f"F1 Macro: {f1_score(y_test, y_pred, average='macro'):.4f}")
if y_test.nunique() == 2:
    auc = roc_auc_score(y_test, y_proba[:, 1])
else:
    auc = 0.5

print(f"ROC AUC:  {auc:.4f}")
print("\n", classification_report(y_test, y_pred))

F1 Macro: 0.3775
ROC AUC:  0.5000

               precision    recall  f1-score   support

           0       0.61      1.00      0.76     20311
           1       0.00      0.00      0.00     13179

    accuracy                           0.61     33490
   macro avg       0.30      0.50      0.38     33490
weighted avg       0.37      0.61      0.46     33490



d:\Education\Arcticle\dtp_project\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Education\Arcticle\dtp_project\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Education\Arcticle\dtp_project\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 